# Notebook 04: Model Evaluation
**Project:** Deep Learning-Based COVID-19 Detection from Chest X-ray Images Using Explainable AI  
**Student:** Channabasavanna Santosh Pawate (16150425)  

## Objective
Comprehensive evaluation of the trained ViT-B/16 model on the held-out test set.

**Metrics:**
- Accuracy, Precision, Recall, F1-score (macro + per-class)
- Confusion matrix
- ROC-AUC curve (one-vs-rest, per class)
- Classification report

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, auc, ConfusionMatrixDisplay
)
import torch
import torch.nn.functional as F
from transformers import ViTForImageClassification
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CLASSES = ['Normal', 'COVID-19', 'Viral Pneumonia']
DATA_ROOT = '../data'
MODEL_PATH = '../model/saved/vit_covid_final'

print(f'Device: {DEVICE}')
print(f'Model path: {MODEL_PATH}')

## 1. Load Model & Test Data

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
IMG_SIZE = 224

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])


class ChestXrayDataset(Dataset):
    CLASSES = ['Normal', 'COVID-19', 'Viral Pneumonia']
    CLASS2IDX = {c: i for i, c in enumerate(CLASSES)}

    def __init__(self, root_dir, transform=None):
        self.transform = transform
        self.samples = []
        for cls in self.CLASSES:
            d = os.path.join(root_dir, cls)
            if not os.path.exists(d):
                continue
            for f in os.listdir(d):
                if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                    self.samples.append((os.path.join(d, f), self.CLASS2IDX[cls]))

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        p, l = self.samples[idx]
        img = Image.open(p).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, l


# Load model
if os.path.exists(MODEL_PATH):
    model = ViTForImageClassification.from_pretrained(MODEL_PATH)
    model = model.to(DEVICE)
    model.eval()
    print('Model loaded successfully')
else:
    print(f'Model not found at {MODEL_PATH}')
    print('Please run Notebook 03 first to train the model.')

# Load test data
if os.path.exists(DATA_ROOT):
    test_ds = ChestXrayDataset(os.path.join(DATA_ROOT, 'test'), val_transform)
    test_loader = DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=4)
    print(f'Test set: {len(test_ds)} images')
else:
    print('Test data not found.')

## 2. Run Inference

In [ ]:
@torch.no_grad()
def get_predictions(model, loader, device):
    all_labels, all_preds, all_probs = [], [], []
    for images, labels in loader:
        images = images.to(device)
        outputs = model(pixel_values=images)
        probs = F.softmax(outputs.logits, dim=1).cpu().numpy()
        preds = probs.argmax(axis=1)
        all_labels.extend(labels.numpy())
        all_preds.extend(preds)
        all_probs.extend(probs)
    return np.array(all_labels), np.array(all_preds), np.array(all_probs)


if 'model' in dir() and 'test_loader' in dir():
    y_true, y_pred, y_prob = get_predictions(model, test_loader, DEVICE)
    acc = (y_true == y_pred).mean()
    print(f'Test Accuracy: {acc*100:.2f}%')
    print(f'Total samples: {len(y_true)}')
else:
    print('Model or data not available. See cells above.')

## 3. Classification Report

In [ ]:
if 'y_true' in dir():
    report = classification_report(y_true, y_pred, target_names=CLASSES, digits=4)
    print('=== Classification Report ===')
    print(report)
    
    # Save
    os.makedirs('../reports', exist_ok=True)
    with open('../reports/classification_report.txt', 'w') as f:
        f.write(f'ViT-B/16 COVID-19 Detection\n')
        f.write(f'Test Accuracy: {acc*100:.2f}%\n\n')
        f.write(report)
    print('\nReport saved to ../reports/classification_report.txt')

## 4. Confusion Matrix

In [ ]:
if 'y_true' in dir():
    cm = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]  # row-normalised
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Raw counts
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASSES)
    disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
    axes[0].set_title('Confusion Matrix (Counts)', fontsize=12, fontweight='bold')
    axes[0].tick_params(axis='x', labelrotation=15)
    
    # Normalised
    im = axes[1].imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
    plt.colorbar(im, ax=axes[1])
    axes[1].set_xticks(range(len(CLASSES)))
    axes[1].set_yticks(range(len(CLASSES)))
    axes[1].set_xticklabels(CLASSES, rotation=15)
    axes[1].set_yticklabels(CLASSES)
    axes[1].set_xlabel('Predicted')
    axes[1].set_ylabel('True')
    axes[1].set_title('Confusion Matrix (Normalised)', fontsize=12, fontweight='bold')
    for i in range(len(CLASSES)):
        for j in range(len(CLASSES)):
            axes[1].text(j, i, f'{cm_norm[i,j]:.2f}',
                         ha='center', va='center',
                         color='white' if cm_norm[i,j] > 0.5 else 'black')
    
    plt.suptitle('ViT-B/16 Confusion Matrix — COVID-19 Detection', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../reports/figures/confusion_matrix.png', bbox_inches='tight', dpi=150)
    plt.show()

## 5. ROC-AUC Curves

In [ ]:
if 'y_true' in dir():
    from sklearn.preprocessing import label_binarize
    
    y_bin = label_binarize(y_true, classes=[0, 1, 2])
    colors = ['#2ecc71', '#e74c3c', '#f39c12']
    
    fig, ax = plt.subplots(figsize=(8, 6))
    
    for i, (cls, color) in enumerate(zip(CLASSES, colors)):
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_prob[:, i])
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, color=color, lw=2, label=f'{cls} (AUC = {roc_auc:.4f})')
    
    ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5, label='Random classifier')
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.set_xlabel('False Positive Rate', fontsize=11)
    ax.set_ylabel('True Positive Rate', fontsize=11)
    ax.set_title('ROC Curves — One-vs-Rest (per class)', fontsize=12, fontweight='bold')
    ax.legend(loc='lower right')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('../reports/figures/roc_curves.png', bbox_inches='tight', dpi=150)
    plt.show()

## 6. Per-Class Performance Summary

In [ ]:
if 'y_true' in dir():
    from sklearn.metrics import precision_score, recall_score, f1_score
    
    metrics = {}
    for i, cls in enumerate(CLASSES):
        y_bin_cls = (y_true == i).astype(int)
        y_pred_cls = (y_pred == i).astype(int)
        fpr, tpr, _ = roc_curve(y_bin_cls, y_prob[:, i])
        metrics[cls] = {
            'Precision': precision_score(y_bin_cls, y_pred_cls),
            'Recall': recall_score(y_bin_cls, y_pred_cls),
            'F1': f1_score(y_bin_cls, y_pred_cls),
            'AUC': auc(fpr, tpr)
        }
    
    import pandas as pd
    df_metrics = pd.DataFrame(metrics).T.round(4)
    print('=== Per-Class Metrics ===')
    print(df_metrics)
    print(f'\nOverall Test Accuracy: {acc*100:.2f}%')
    print(f'Macro F1: {f1_score(y_true, y_pred, average="macro"):.4f}')
    
    # Bar chart
    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(CLASSES))
    width = 0.2
    metric_names = ['Precision', 'Recall', 'F1', 'AUC']
    metric_colors = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6']
    
    for j, (metric, color) in enumerate(zip(metric_names, metric_colors)):
        vals = [metrics[cls][metric] for cls in CLASSES]
        ax.bar(x + j * width - 1.5 * width, vals, width, label=metric, color=color, alpha=0.85)
    
    ax.set_xlabel('Class')
    ax.set_ylabel('Score')
    ax.set_title('Per-Class Performance Metrics', fontsize=12, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(CLASSES)
    ax.set_ylim([0, 1.1])
    ax.legend()
    ax.axhline(y=0.95, color='black', linestyle=':', alpha=0.5, label='95% target')
    ax.grid(True, alpha=0.2, axis='y')
    plt.tight_layout()
    plt.savefig('../reports/figures/per_class_metrics.png', bbox_inches='tight', dpi=150)
    plt.show()

## 7. Evaluation Summary for Dissertation

| Metric | Value |
|---|---|
| Test Accuracy | (see above) |
| Macro F1 | (see above) |
| COVID-19 AUC | (see above) |
| Normal AUC | (see above) |
| Pneumonia AUC | (see above) |

**Target:** >95% accuracy, consistent with Zhang et al. (2023) and Chowdhury et al. (2020) benchmarks.

**Next: Run `model/xai.py` for attention heatmap generation.**